# NOTEBOOK 02 — build base features

Цель:
- собрать базовую фичевую матрицу для следующих этапов:
  - L1 OOF LGBM
  - final LGBM meta
  - final CatBoost meta

Что делаем:
- читаем артефакты из `NOTEBOOK 00` и `NOTEBOOK 01`;
- берем:
  - `main categorical`
  - `main numeric`
  - `top-700 extra`
- добавляем:
  - `null_count_main`
  - `null_count_extra`
  - `null_ratio_main`
  - `null_ratio_extra`
  - row stats по main / extra:
    - mean
    - std
    - min
    - max
    - nonzero_count
  - frequency encoding для main categorical
- кодируем `main categorical` в стабильные `Int32` codes
- максимально ужимаем типы данных

Что сохраняем:
- `artifacts/features/train_base_features.parquet`
- `artifacts/features/test_base_features.parquet`
- `artifacts/features/base_feature_cols.json`

In [1]:
# =========================
# Imports
# =========================
from pathlib import Path
import gc
import json
import warnings

import numpy as np
import polars as pl

warnings.filterwarnings("ignore")

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)

polars.config.Config

In [4]:
# =========================
# Paths
# =========================
DATA_DIR = Path("/kaggle/input/datasets/hatab123/data-fusion-contest-2026/")
PREPARE_DIR01 = Path("/kaggle/input/notebooks/viktoriasvetankova/00-schema-and-folds2803/prepared")
PREPARE_DIR02 = Path("/kaggle/input/notebooks/viktoriasvetankova/01-feature-selection-41targets2803/prepared")
WORK_DIR = Path("/kaggle/working/prepared/artifacts")
WORK_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "train_main": DATA_DIR / "train_main_features.parquet",
    "train_extra": DATA_DIR / "train_extra_features.parquet",
    "train_target": DATA_DIR / "train_target.parquet",
    "test_main": DATA_DIR / "test_main_features.parquet",
    "test_extra": DATA_DIR / "test_extra_features.parquet",
    "sample_submit": DATA_DIR / "sample_submit.parquet",
}

ARTIFACTS_DIR01 = PREPARE_DIR01 / "artifacts"
ARTIFACTS_DIR02 = PREPARE_DIR02 / "artifacts"
CONFIG_DIR = ARTIFACTS_DIR01 / "config"
SELECTED_DIR = ARTIFACTS_DIR02 / "selected_features"
FEATURES_DIR = WORK_DIR / "features"
LOG_DIR = WORK_DIR / "logs"

FEATURES_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("WORK_DIR:", WORK_DIR)
for name, path in PATHS.items():
    print(f"{name:>14}: {path} | exists={path.exists()}")

print("\nArtifacts:")
for p in [CONFIG_DIR, SELECTED_DIR, FEATURES_DIR]:
    print(p, "| exists=", p.exists())

DATA_DIR: /kaggle/input/datasets/hatab123/data-fusion-contest-2026
WORK_DIR: /kaggle/working/prepared/artifacts
    train_main: /kaggle/input/datasets/hatab123/data-fusion-contest-2026/train_main_features.parquet | exists=True
   train_extra: /kaggle/input/datasets/hatab123/data-fusion-contest-2026/train_extra_features.parquet | exists=True
  train_target: /kaggle/input/datasets/hatab123/data-fusion-contest-2026/train_target.parquet | exists=True
     test_main: /kaggle/input/datasets/hatab123/data-fusion-contest-2026/test_main_features.parquet | exists=True
    test_extra: /kaggle/input/datasets/hatab123/data-fusion-contest-2026/test_extra_features.parquet | exists=True
 sample_submit: /kaggle/input/datasets/hatab123/data-fusion-contest-2026/sample_submit.parquet | exists=True

Artifacts:
/kaggle/input/notebooks/viktoriasvetankova/00-schema-and-folds2803/prepared/artifacts/config | exists= True
/kaggle/input/notebooks/viktoriasvetankova/01-feature-selection-41targets2803/prepared/arti

In [5]:
# =========================
# Config
# =========================
ID_COL = "customer_id"
FLOAT_DTYPE = pl.Float32
INT_DTYPE = pl.Int32
BOOL_INT_DTYPE = pl.Int8
FREQ_DTYPE = pl.UInt32

TRAIN_OUT = FEATURES_DIR / "train_base_features.parquet"
TEST_OUT = FEATURES_DIR / "test_base_features.parquet"
META_OUT = FEATURES_DIR / "base_feature_cols.json"

In [6]:
# =========================
# Helpers
# =========================
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(path: Path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def report_mem(df: pl.DataFrame, name: str):
    mem_mb = df.estimated_size() / 1024**2
    print(f"{name}: shape={df.shape}, memory={mem_mb:,.2f} MB")

def downcast_pl(df: pl.DataFrame) -> pl.DataFrame:
    exprs = []
    for col, dtype in zip(df.columns, df.dtypes):
        if dtype == pl.Float64:
            exprs.append(pl.col(col).cast(pl.Float32))
        elif dtype == pl.Int64:
            exprs.append(pl.col(col).cast(pl.Int32))
        elif dtype == pl.UInt64:
            exprs.append(pl.col(col).cast(pl.UInt32))
        elif dtype == pl.UInt16:
            exprs.append(pl.col(col).cast(pl.UInt16))
        elif dtype == pl.UInt8:
            exprs.append(pl.col(col).cast(pl.UInt8))
        else:
            exprs.append(pl.col(col))
    return df.select(exprs)

def expr_nonzero_count(cols):
    return pl.sum_horizontal([
        ((pl.col(c).fill_null(0) != 0).cast(pl.Int16))
        for c in cols
    ])

def build_row_stats(df: pl.DataFrame, cols, prefix: str) -> pl.DataFrame:
    if len(cols) == 0:
        return pl.DataFrame({
            f"{prefix}_mean": [],
            f"{prefix}_std": [],
            f"{prefix}_min": [],
            f"{prefix}_max": [],
            f"{prefix}_nonzero_count": [],
        })

    count_expr = pl.sum_horizontal([
        pl.col(c).is_not_null().cast(pl.Int16) for c in cols
    ]).alias(f"{prefix}_nonnull_count")

    sum_expr = pl.sum_horizontal([
        pl.col(c).fill_null(0).cast(pl.Float32) for c in cols
    ]).alias(f"{prefix}_sum")

    sumsq_expr = pl.sum_horizontal([
        (pl.col(c).fill_null(0).cast(pl.Float32) ** 2) for c in cols
    ]).alias(f"{prefix}_sumsq")

    tmp = df.select([
        count_expr,
        sum_expr,
        sumsq_expr,
        pl.min_horizontal([pl.col(c) for c in cols]).cast(pl.Float32).alias(f"{prefix}_min"),
        pl.max_horizontal([pl.col(c) for c in cols]).cast(pl.Float32).alias(f"{prefix}_max"),
        expr_nonzero_count(cols).cast(pl.Int16).alias(f"{prefix}_nonzero_count"),
    ])

    tmp = tmp.with_columns([
        pl.when(pl.col(f"{prefix}_nonnull_count") > 0)
          .then((pl.col(f"{prefix}_sum") / pl.col(f"{prefix}_nonnull_count")).cast(pl.Float32))
          .otherwise(None)
          .alias(f"{prefix}_mean"),
    ])

    tmp = tmp.with_columns([
        pl.when(pl.col(f"{prefix}_nonnull_count") > 0)
          .then(
              (
                  (pl.col(f"{prefix}_sumsq") / pl.col(f"{prefix}_nonnull_count"))
                  - (pl.col(f"{prefix}_mean") ** 2)
              ).clip(lower_bound=0).sqrt().cast(pl.Float32)
          )
          .otherwise(None)
          .alias(f"{prefix}_std")
    ])

    return tmp.select([
        f"{prefix}_mean",
        f"{prefix}_std",
        f"{prefix}_min",
        f"{prefix}_max",
        f"{prefix}_nonzero_count",
    ])

def build_null_features(df: pl.DataFrame, cols, prefix: str) -> pl.DataFrame:
    n = max(len(cols), 1)
    return df.select([
        pl.sum_horizontal([pl.col(c).is_null().cast(pl.Int16) for c in cols]).cast(pl.Int16).alias(f"null_count_{prefix}"),
        (
            pl.sum_horizontal([pl.col(c).is_null().cast(pl.Int16) for c in cols]).cast(pl.Float32) / pl.lit(float(n))
        ).cast(pl.Float32).alias(f"null_ratio_{prefix}")
    ])

def normalize_cat_expr(col_name: str):
    # единое представление для train/test и для null
    return pl.col(col_name).cast(pl.Utf8).fill_null("__MISSING__").alias(col_name)

def make_code_and_freq_maps(train_df: pl.DataFrame, test_df: pl.DataFrame, cat_cols):
    code_maps = {}
    freq_maps = {}
    for i, col in enumerate(cat_cols, start=1):
        print(f"[{i:03d}/{len(cat_cols)}] preparing cat maps for {col}")
        combined = pl.concat(
            [
                train_df.select(normalize_cat_expr(col)),
                test_df.select(normalize_cat_expr(col)),
            ],
            how="vertical_relaxed",
        )

        code_map = (
            combined
            .unique(maintain_order=True)
            .with_row_index(name=f"{col}__code")
            .with_columns(pl.col(f"{col}__code").cast(pl.Int32))
        )
        freq_map = (
            combined
            .group_by(col)
            .len()
            .rename({"len": f"{col}__freq"})
            .with_columns(pl.col(f"{col}__freq").cast(pl.UInt32))
        )

        code_maps[col] = code_map
        freq_maps[col] = freq_map

        del combined
        gc.collect()
    return code_maps, freq_maps

def apply_cat_maps(df: pl.DataFrame, cat_cols, code_maps, freq_maps):
    out = df
    for i, col in enumerate(cat_cols, start=1):
        print(f"[{i:03d}/{len(cat_cols)}] applying cat maps for {col}")
        out = (
            out
            .with_columns(normalize_cat_expr(col))
            .join(code_maps[col], on=col, how="left")
            .join(freq_maps[col], on=col, how="left")
            .drop(col)
            .with_columns([
                pl.col(f"{col}__code").cast(pl.Int32),
                pl.col(f"{col}__freq").cast(pl.UInt32),
            ])
        )
        gc.collect()
    return out

def load_selected_lists():
    main_cat_cols = load_json(CONFIG_DIR / "main_cat_cols.json")
    main_num_cols = load_json(CONFIG_DIR / "main_num_cols.json")
    extra_top700 = load_json(SELECTED_DIR / "selected_extra_top700.json")
    extra_top300 = load_json(SELECTED_DIR / "selected_extra_top300.json")
    target_cols = load_json(CONFIG_DIR / "target_cols.json")
    return main_cat_cols, main_num_cols, extra_top700, extra_top300, target_cols

In [7]:
# =========================
# Load config artifacts from Notebook 00 and Notebook 01
# =========================
main_cat_cols, main_num_cols, extra_top700, extra_top300, target_cols = load_selected_lists()

print("main_cat_cols :", len(main_cat_cols))
print("main_num_cols :", len(main_num_cols))
print("extra_top700  :", len(extra_top700))
print("extra_top300  :", len(extra_top300))
print("target_cols   :", len(target_cols))

assert set(extra_top300).issubset(set(extra_top700)), "top300 must be subset of top700"

main_cat_cols : 67
main_num_cols : 132
extra_top700  : 700
extra_top300  : 300
target_cols   : 41


## 1. Load main blocks

In [8]:
train_main_cols = [ID_COL] + main_cat_cols + main_num_cols
test_main_cols = [ID_COL] + main_cat_cols + main_num_cols

train_main = pl.read_parquet(PATHS["train_main"], columns=train_main_cols)
test_main = pl.read_parquet(PATHS["test_main"], columns=test_main_cols)

train_main = downcast_pl(train_main)
test_main = downcast_pl(test_main)

report_mem(train_main, "train_main")
report_mem(test_main, "test_main")

train_main: shape=(750000, 200), memory=584.01 MB
test_main: shape=(250000, 200), memory=194.67 MB


## 2. Build null features and row stats for main block

In [9]:
main_all_feature_cols = main_cat_cols + main_num_cols

train_main_null = build_null_features(train_main, main_all_feature_cols, "main")
test_main_null = build_null_features(test_main, main_all_feature_cols, "main")

train_main_stats = build_row_stats(train_main, main_num_cols, "main")
test_main_stats = build_row_stats(test_main, main_num_cols, "main")

report_mem(train_main_null, "train_main_null")
report_mem(train_main_stats, "train_main_stats")

train_main_null: shape=(750000, 2), memory=4.29 MB
train_main_stats: shape=(750000, 5), memory=13.32 MB


## 3. Encode main categorical + frequency encoding

In [10]:
code_maps, freq_maps = make_code_and_freq_maps(train_main, test_main, main_cat_cols)

train_main_encoded = apply_cat_maps(train_main, main_cat_cols, code_maps, freq_maps)
test_main_encoded = apply_cat_maps(test_main, main_cat_cols, code_maps, freq_maps)

report_mem(train_main_encoded, "train_main_encoded")
report_mem(test_main_encoded, "test_main_encoded")

# free raw cat maps from memory only after both train/test transformed
gc.collect()

[001/67] preparing cat maps for cat_feature_1
[002/67] preparing cat maps for cat_feature_2
[003/67] preparing cat maps for cat_feature_3
[004/67] preparing cat maps for cat_feature_4
[005/67] preparing cat maps for cat_feature_5
[006/67] preparing cat maps for cat_feature_6
[007/67] preparing cat maps for cat_feature_7
[008/67] preparing cat maps for cat_feature_8
[009/67] preparing cat maps for cat_feature_9
[010/67] preparing cat maps for cat_feature_10
[011/67] preparing cat maps for cat_feature_11
[012/67] preparing cat maps for cat_feature_12
[013/67] preparing cat maps for cat_feature_13
[014/67] preparing cat maps for cat_feature_14
[015/67] preparing cat maps for cat_feature_15
[016/67] preparing cat maps for cat_feature_16
[017/67] preparing cat maps for cat_feature_17
[018/67] preparing cat maps for cat_feature_18
[019/67] preparing cat maps for cat_feature_19
[020/67] preparing cat maps for cat_feature_20
[021/67] preparing cat maps for cat_feature_21
[022/67] preparing cat

0

In [11]:
train_main_encoded

customer_id,num_feature_1,num_feature_2,num_feature_3,num_feature_4,num_feature_5,num_feature_6,num_feature_7,num_feature_8,num_feature_9,…,cat_feature_63__code,cat_feature_63__freq,cat_feature_64__code,cat_feature_64__freq,cat_feature_65__code,cat_feature_65__freq,cat_feature_66__code,cat_feature_66__freq,cat_feature_67__code,cat_feature_67__freq
i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,i32,u32,i32,u32,i32,u32,i32,u32,i32,u32
1000001,-0.038834,null,-0.157258,null,-0.088059,-0.307721,-0.147601,-0.257746,0.0,…,0,380556,0,636202,0,781594,0,444723,0,636202
1000002,-0.038834,-0.210019,-0.286426,null,-0.088059,-0.307721,4.082561,-0.257746,0.0,…,0,380556,1,363686,0,781594,1,344215,1,363014
1000003,-0.038834,-0.210019,null,null,-0.088059,-0.307721,-0.147601,-0.257746,0.0,…,0,380556,1,363686,0,781594,0,444723,1,363014
1000004,-0.038834,null,null,null,-0.088059,-0.307721,-0.147601,-0.257746,0.0,…,1,461022,0,636202,0,781594,1,344215,0,636202
1000005,-0.038834,-0.210019,null,null,-0.088059,-0.307721,-0.147601,1.418979,0.0,…,2,158422,1,363686,1,158422,0,444723,1,363014
1000006,-0.038834,null,null,null,-0.088059,-0.307721,-0.147601,-0.257746,0.0,…,0,380556,0,636202,0,781594,0,444723,0,636202
1000007,-0.038834,null,null,null,-0.088059,-0.307721,-0.147601,null,0.0,…,2,158422,0,636202,1,158422,1,344215,0,636202
1000008,null,null,null,null,null,null,null,null,null,…,0,380556,0,636202,0,781594,1,344215,0,636202
1000009,-0.038834,-0.210019,null,null,-0.088059,6.54352,0.90994,-0.257746,0.0,…,0,380556,0,636202,0,781594,1,344215,0,636202


## 4. Load top-700 extra

In [12]:
train_extra_cols = [ID_COL] + extra_top700
test_extra_cols = [ID_COL] + extra_top700

train_extra = pl.read_parquet(PATHS["train_extra"], columns=train_extra_cols)
test_extra = pl.read_parquet(PATHS["test_extra"], columns=test_extra_cols)

train_extra = downcast_pl(train_extra)
test_extra = downcast_pl(test_extra)

report_mem(train_extra, "train_extra_top700")
report_mem(test_extra, "test_extra_top700")

train_extra_top700: shape=(750000, 701), memory=2,068.16 MB
test_extra_top700: shape=(250000, 701), memory=689.39 MB


## 5. Build null features and row stats for extra block

In [13]:
train_extra_null = build_null_features(train_extra, extra_top700, "extra")
test_extra_null = build_null_features(test_extra, extra_top700, "extra")

train_extra_stats = build_row_stats(train_extra, extra_top700, "extra")
test_extra_stats = build_row_stats(test_extra, extra_top700, "extra")

report_mem(train_extra_null, "train_extra_null")
report_mem(train_extra_stats, "train_extra_stats")

train_extra_null: shape=(750000, 2), memory=4.29 MB
train_extra_stats: shape=(750000, 5), memory=13.32 MB


## 6. Assemble final base train/test matrices

In [15]:
# keep ids for alignment checks
train_ids_main = train_main_encoded.get_column(ID_COL)
test_ids_main = test_main_encoded.get_column(ID_COL)
train_ids_extra = train_extra.get_column(ID_COL)
test_ids_extra = test_extra.get_column(ID_COL)

assert train_ids_main.equals(train_ids_extra), "train main / extra id mismatch"
assert test_ids_main.equals(test_ids_extra), "test main / extra id mismatch"

# drop ids before hstack; keep one final customer_id
train_main_noid = train_main_encoded.drop(ID_COL)
test_main_noid = test_main_encoded.drop(ID_COL)

train_extra_noid = train_extra.drop(ID_COL)
test_extra_noid = test_extra.drop(ID_COL)

train_base = pl.concat(
    [
        pl.DataFrame({ID_COL: train_ids_main}),
        train_main_noid,
        train_extra_noid,
        train_main_null,
        train_extra_null,
        train_main_stats,
        train_extra_stats,
    ],
    how="horizontal",
)

test_base = pl.concat(
    [
        pl.DataFrame({ID_COL: test_ids_main}),
        test_main_noid,
        test_extra_noid,
        test_main_null,
        test_extra_null,
        test_main_stats,
        test_extra_stats,
    ],
    how="horizontal",
)

train_base = downcast_pl(train_base)
test_base = downcast_pl(test_base)

report_mem(train_base, "train_base")
report_mem(test_base, "test_base")
print("train_base columns:", len(train_base.columns))
print("test_base columns :", len(test_base.columns))

train_base: shape=(750000, 981), memory=2,876.22 MB
test_base: shape=(250000, 981), memory=958.68 MB
train_base columns: 981
test_base columns : 981


In [ ]:
train_base.columns

## 7. Save artifacts

In [16]:
main_cat_code_cols = [f"{c}__code" for c in main_cat_cols]
main_cat_freq_cols = [f"{c}__freq" for c in main_cat_cols]

null_feature_cols = [
    "null_count_main", "null_ratio_main",
    "null_count_extra", "null_ratio_extra",
]

row_stat_cols = [
    "main_mean", "main_std", "main_min", "main_max", "main_nonzero_count",
    "extra_mean", "extra_std", "extra_min", "extra_max", "extra_nonzero_count",
]

all_base_feature_cols = [c for c in train_base.columns if c != ID_COL]

meta = {
    "id_col": ID_COL,
    "target_cols": target_cols,
    "main_num_cols": main_num_cols,
    "main_cat_raw_cols": main_cat_cols,
    "main_cat_code_cols": main_cat_code_cols,
    "main_cat_freq_cols": main_cat_freq_cols,
    "extra_top700_cols": extra_top700,
    "extra_top300_cols": extra_top300,
    "null_feature_cols": null_feature_cols,
    "row_stat_cols": row_stat_cols,
    "all_base_feature_cols": all_base_feature_cols,
}

train_base.write_parquet(TRAIN_OUT)
test_base.write_parquet(TEST_OUT)
save_json(META_OUT, meta)

print("Saved:")
print(" -", TRAIN_OUT)
print(" -", TEST_OUT)
print(" -", META_OUT)

Saved:
 - /kaggle/working/prepared/artifacts/features/train_base_features.parquet
 - /kaggle/working/prepared/artifacts/features/test_base_features.parquet
 - /kaggle/working/prepared/artifacts/features/base_feature_cols.json


In [17]:
print("Sample of train_base:")
train_base.head(3)

Sample of train_base:


customer_id,num_feature_1,num_feature_2,num_feature_3,num_feature_4,num_feature_5,num_feature_6,num_feature_7,num_feature_8,num_feature_9,…,main_mean,main_std,main_min,main_max,main_nonzero_count,extra_mean,extra_std,extra_min,extra_max,extra_nonzero_count
i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,i16,f32,f32,f32,f32,i16
1000001,-0.038834,null,-0.157258,null,-0.088059,-0.307721,-0.147601,-0.257746,0.0,…,-0.090848,0.280466,-0.542784,1.61104,68,-0.063727,0.546359,-0.919307,5.265816,304
1000002,-0.038834,-0.210019,-0.286426,null,-0.088059,-0.307721,4.082561,-0.257746,0.0,…,0.062574,0.635684,-0.805771,4.082561,85,0.056438,0.622135,-1.785514,5.069804,408
1000003,-0.038834,-0.210019,null,null,-0.088059,-0.307721,-0.147601,-0.257746,0.0,…,-0.168482,0.209082,-0.983056,0.427056,75,-0.055926,0.967207,-1.659281,11.58178,336


In [18]:
# =========================
# Cleanup
# =========================
del train_main, test_main, train_main_encoded, test_main_encoded
del train_extra, test_extra
del train_main_null, test_main_null, train_extra_null, test_extra_null
del train_main_stats, test_main_stats, train_extra_stats, test_extra_stats
del train_main_noid, test_main_noid, train_extra_noid, test_extra_noid
gc.collect()

print("Notebook 02 finished successfully.")

Notebook 02 finished successfully.
